In [1]:
%matplotlib Qt

In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from astropy.io import fits
from sklearn.model_selection import train_test_split
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
from tensorflow import keras
import keras_tuner as kt

I0000 00:00:1784989961.572785   10299 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784989961.777329   10299 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784989962.886957   10299 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
# Check if TensorFlow detects any GPUs
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"Success! TensorFlow detected {len(gpus)} GPU(s): {gpus}")

    # Best Practice: Enable Memory Growth
    # This prevents TensorFlow from instantly reserving all your GPU VRAM,
    # which can crash your system or prevent other apps from running.
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)
else:
    print("Warning: No GPU detected. TensorFlow will fall back to the CPU.")

E0000 00:00:1784989971.471093   10299 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [4]:
@tf.keras.utils.register_keras_serializable(package="Custom")
class PearsonHuberLoss(tf.keras.losses.Loss):
    """
    Combines Huber Loss (robust point-wise distance) with Pearson Correlation Loss.
    Pearson Correlation Loss penalizes regression-to-the-mean (horizontal flattening)
    by forcing predictions to maintain linear correlation with true values.
    """
    def __init__(self, alpha=0.25, delta=0.1, name="pearson_huber_loss", reduction="sum_over_batch_size"):
        super().__init__(name=name, reduction=reduction)
        self.alpha = alpha
        self.delta = delta
        self.huber = tf.keras.losses.Huber(delta=delta, reduction=reduction)

    def call(self, y_true, y_pred):
        # 1. Huber Loss (Point-wise error)
        h_loss = self.huber(y_true, y_pred)

        # 2. Pearson Correlation Loss across the current batch
        y_true_mean = tf.reduce_mean(y_true, axis=0, keepdims=True)
        y_pred_mean = tf.reduce_mean(y_pred, axis=0, keepdims=True)

        y_true_centered = y_true - y_true_mean
        y_pred_centered = y_pred - y_pred_mean

        covariance = tf.reduce_sum(y_true_centered * y_pred_centered, axis=0)
        var_true = tf.reduce_sum(tf.square(y_true_centered), axis=0)
        var_pred = tf.reduce_sum(tf.square(y_pred_centered), axis=0)

        # Small epsilon to prevent division by zero
        std_product = tf.sqrt(var_true * var_pred + 1e-7)
        pearson_r = covariance / std_product

        # We want to maximize pearson_r (ideal = 1.0), so we minimize (1.0 - pearson_r)
        p_loss = tf.reduce_mean(1.0 - pearson_r)

        return h_loss + self.alpha * p_loss

    def get_config(self):
        config = super().get_config()
        config.update({
            "alpha": self.alpha,
            "delta": self.delta
        })
        return config

In [5]:
def load_and_preprocess_regression_data(data_dir, csv_path, filename_col, ba_col, ca_col):
    """
    Reads FITS files and regression targets (b/a, c/a).
    Applies global robust scaling to preserve physical scales.
    """
    df = pd.read_csv(csv_path)

    raw_images = []
    targets = []

    print("Reading FITS files and handling NaNs...")
    for index, row in df.iterrows():
        filename = row[filename_col]
        # Our targets are now two continuous values: [b/a, c/a]
        target_vals = [row[ba_col], row[ca_col]]

        filepath = os.path.join(data_dir, filename)

        with fits.open(filepath) as hdul:
            data = hdul[0].data  # Expected shape: (3, 69, 69)

        # Handle NaNs and apply individual scaling for each channel
        for i in range(3):
            channel_data = data[i, :, :]
            mean_val = np.nanmean(channel_data)
            if np.isnan(mean_val):
                mean_val = 0.0
            channel_data[np.isnan(channel_data)] = mean_val

            channel_data = channel_data / np.nanmax(np.abs(channel_data))

            data[i, :, :] = channel_data

        # Transpose to (Height, Width, Channels) -> (69, 69, 3)
        data = np.transpose(data, (1, 2, 0))
        raw_images.append(data)
        targets.append(target_vals)

    X = np.array(raw_images, dtype=np.float32)
    y = np.array(targets, dtype=np.float32)

    return X, y

In [6]:
# --- CONFIGURATION ---
DATA_DIRECTORY = "./VELOCITY_VDISP_FLUX_MAPS"
CSV_FILE_PATH = "./files_list_and_axis.csv"
FILENAME_COLUMN_NAME = "filename"
BA_COLUMN_NAME = "b_a"        # Update with your actual column name for b/a
CA_COLUMN_NAME = "c_a"        # Update with your actual column name for c/a

In [7]:
# 1. Load Data
X, y = load_and_preprocess_regression_data(
    data_dir=DATA_DIRECTORY,
    csv_path=CSV_FILE_PATH,
    filename_col=FILENAME_COLUMN_NAME,
    ba_col=BA_COLUMN_NAME,
    ca_col=CA_COLUMN_NAME
)

Reading FITS files and handling NaNs...


In [8]:
def compute_sample_weights(y):
    """
    Calculates sample weights based on the 2D density of the targets.
    Rare (b/a, c/a) combinations will receive higher weights.
    """
    print("Calculating 2D Kernel Density for sample weighting...")

    # Transpose y for KDE: needs shape (2, num_samples)
    values = y.T

    # Calculate the density of the parameter space using Gaussian KDE
    kde = gaussian_kde(values)
    density = kde.evaluate(values)

    # Inverse density weighting: rarer points get higher weights
    # We add a small constant (epsilon) to prevent division by zero or infinite weights
    epsilon = np.percentile(density, 5) # Prevent extreme spikes for utter outliers
    weights = 1.0 / (density + epsilon)

    # Normalize weights so the mean weight is 1.0.
    # This prevents the overall learning rate from needing massive adjustment.
    weights = weights / np.mean(weights)

    return weights.astype(np.float32)

In [9]:
# 2. Split Data (No stratification needed for continuous regression)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Calculate Sample Weights for the Training Set
# This applies the "penalty" for rare parameters
train_weights = compute_sample_weights(y_train)

Calculating 2D Kernel Density for sample weighting...


In [10]:
# ==========================================
# 3. REGRESSION MODEL ARCHITECTURE
# ==========================================

def build_regression_model(input_shape=(69, 69, 3)):
    """
    Builds a CNN optimized for continuous parameter extraction.
    Ends with a Sigmoid layer to strictly bound predictions between 0 and 1.
    """
    l2_reg = tf.keras.regularizers.l2(1e-4)

    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        # Spatial Augmentations
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),

        # DEPTHWISE CONVOLUTION (Independent physical maps)
        tf.keras.layers.DepthwiseConv2D(
            kernel_size=(3, 3),
            depth_multiplier=8,
            activation='relu',
            padding='same',
            depthwise_regularizer=l2_reg
        ),
        tf.keras.layers.Conv2D(32, (1, 1), activation='relu', kernel_regularizer=l2_reg),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

        # MIXING & FEATURE EXTRACTION
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', kernel_regularizer=l2_reg),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=l2_reg),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

        # We add one more Convolution block to distill features for regression
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=l2_reg),

        # FLATTEN & REGRESS
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=l2_reg),
        tf.keras.layers.Dropout(0.3),

        # OUTPUT LAYER: 2 units (b/a, c/a)
        # Activation is 'sigmoid' which mathematically forces the output
        # to always be strictly between 0.0 and 1.0.
        tf.keras.layers.Dense(2, activation='sigmoid')
    ])

    # We use Mean Squared Error (MSE) for regression.
    # Mean Absolute Error (MAE) is tracked as a secondary metric for interpretability.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0008),
        loss=PearsonHuberLoss(alpha=0.25, delta=0.1),
        metrics=['mae']
    )

    return model

In [ ]:
# ==========================================
# 3. REGRESSION MODEL ARCHITECTURE
# ==========================================

def build_tunable_cnn_model(hp):
    """
    Builds an UNTRAINED CNN model where architectural structural parameters
    are defined dynamically using KerasTuner HyperParameters (hp).
    """
    l2_reg = tf.keras.regularizers.l2(1e-4)

    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(shape=(69, 69, 3)))

    # --- 2. Data Augmentation ---
    model.add(tf.keras.layers.RandomFlip("horizontal_and_vertical"))

    # --- 3. Vary Depthwise Conv Kernel Size & Multiplier ---
    depth_multiplier = hp.Choice('depth_multiplier', values=[4, 8, 16])
    model.add(tf.keras.layers.DepthwiseConv2D(
        kernel_size=(3, 3),
        depth_multiplier=depth_multiplier,
        activation='relu',
        padding='same',
        depthwise_regularizer=l2_reg
    ))

    # --- 4. Vary Conv2D Kernel Sizes & Feature Extraction ---
    conv1_filters = hp.Choice('conv1_filters', values=[16, 32, 64])
    model.add(tf.keras.layers.Conv2D(
        conv1_filters,
        (1, 1),
        activation='relu',
        padding='same',
        kernel_regularizer=l2_reg
    ))
    # Vary pooling strategy (2x2 Max Pooling vs 2x2 Average Pooling)
    pool1_type = hp.Choice('pool1_type', values=['max', 'avg'])
    if pool1_type == 'max':
        model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
    else:
        model.add(tf.keras.layers.AveragePooling2D(pool_size=(2, 2)))

    conv2_kernel = hp.Choice('conv2_kernel_size', values=[3, 5])
    model.add(tf.keras.layers.Conv2D(
        conv1_filters, #same size, because conv 1 has kernel=1
        (conv2_kernel, conv2_kernel),
        activation='relu',
        padding='same',
        kernel_regularizer=l2_reg
    ))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    # --- 6. Stage 4: Deep Feature Extraction Conv2D ---
    conv3_kernel = hp.Choice('conv3_kernel_size', values=[3, 5])
    model.add(tf.keras.layers.Conv2D(
        conv1_filters*2,
        (conv3_kernel, conv3_kernel),
        activation='relu',
        padding='same',
        kernel_regularizer=l2_reg
    ))

    pool3_type = hp.Choice('pool3_type', values=['max', 'none'])
    if pool3_type == 'max':
        model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    # --- 5. Vary Dense Head properties ---
    model.add(tf.keras.layers.GlobalAveragePooling2D())

    dense_units = hp.Choice('dense_units', values=[32, 64, 128])
    model.add(tf.keras.layers.Dense(dense_units, activation='relu', kernel_regularizer=l2_reg))

    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1)
    model.add(tf.keras.layers.Dropout(dropout_rate))

    # --- Output layer: [b/a, delta] ---
    model.add(tf.keras.layers.Dense(2, activation='sigmoid'))

    # --- 6. Vary Learning Rate & Loss Alpha Weight ---
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
    pearson_alpha = hp.Float('pearson_alpha', min_value=0.1, max_value=0.5, step=0.1)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=PearsonHuberLoss(alpha=pearson_alpha, delta=0.1),
        metrics=['mae']
    )

    return model

In [13]:
def run_hyperparameter_search(X_train, y_train, X_val, y_val, max_trials=20):
    """
    Configures Bayesian Optimization tuner to find the best CNN parameters.
    """
    tuner = kt.BayesianOptimization(
        hypermodel=build_tunable_cnn_model,
        objective=kt.Objective("val_loss", direction="min"),
        max_trials=max_trials,
        directory="keras_tuner_dir",
        project_name="ifu_cnn_optimization",
        overwrite=True
    )

    # Early stopping callback to terminate poorly performing trials quickly
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=40,
        restore_best_weights=True
    )

    train_weights = compute_sample_weights(y_train)

    print("\n--- Starting KerasTuner Search ---")
    tuner.search_space_summary()

    tuner.search(
        X_train, y_train,
        sample_weight=train_weights,
        validation_data=(X_val, y_val),
        epochs=300,
        batch_size=32,
        callbacks=[early_stopping]
    )

    print("\n--- Tuning Complete! Best Hyperparameters: ---")
    best_hparams = tuner.get_best_hyperparameters(num_trials=1)[0]

    for param in best_hparams.values:
        print(f"  {param}: {best_hparams.get(param)}")

    # Get the best trained model directly from the tuner
    best_model = tuner.get_best_models(num_models=1)[0]
    return best_model, best_hparams

In [17]:
# Run the hyperparameter search on the untrained model design
best_cnn_model, best_params = run_hyperparameter_search(X_train, y_train, X_val, y_val, max_trials=50)

# Save the best model found
best_cnn_model.save("./best_tuned_ifu_cnn.keras")
print("Best tuned CNN saved to './best_tuned_ifu_cnn.keras'")

Trial 50 Complete [00h 16m 15s]
val_loss: 0.03383410722017288

Best val_loss So Far: 0.030470697209239006
Total elapsed time: 08h 54m 21s

--- Tuning Complete! Best Hyperparameters: ---
  depth_multiplier: 16
  conv1_filters: 64
  pool1_type: max
  conv2_kernel_size: 3
  conv3_kernel_size: 5
  pool3_type: max
  dense_units: 128
  dropout_rate: 0.1
  learning_rate: 0.0005263457974195464
  pearson_alpha: 0.1
Best tuned CNN saved to './best_tuned_ifu_cnn.keras'


/home/astrolander/.generalvenv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [10]:
# 4. Create tf.data.Datasets
batch_size = 32

# Notice we pass a tuple of THREE items here: (features, targets, weights)
# TensorFlow will automatically use the weights to scale the MSE loss!
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train, train_weights))
train_dataset = train_dataset.shuffle(buffer_size=len(X_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Validation dataset does not use weights. We want pure, unweighted MSE for validation evaluation.
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [11]:

# 5. Build Model & Train
model = build_regression_model(input_shape=(69, 69, 3))

I0000 00:00:1784901917.851866   14212 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2987 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [12]:
early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=150,
        restore_best_weights=True
    )

In [13]:
print("Starting training with continuous density weighting...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=1500,
    callbacks=[early_stopping]
)

Starting training with continuous density weighting...
Epoch 1/1500


I0000 00:00:1784901921.264699   14333 service.cc:153] XLA service 0x7b26b80359b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784901921.264715   14333 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1784901921.302348   14333 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1784901921.504883   14333 cuda_dnn.cc:461] Loaded cuDNN version 91900
I0000 00:00:1784901921.550213   14333 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3422__.55
I0000 00:00:1784901922.063326   14333 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set i

 22/251 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2887 - mae: 0.2884

I0000 00:00:1784901928.125083   14333 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


249/251 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2623 - mae: 0.2434

I0000 00:00:1784901930.680341   14333 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3422__.55


251/251 ━━━━━━━━━━━━━━━━━━━━ 19s 44ms/step - loss: 0.2200 - mae: 0.1870 - val_loss: 0.1736 - val_mae: 0.1153
Epoch 2/1500
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.1713 - mae: 0.1205 - val_loss: 0.1626 - val_mae: 0.1086
Epoch 3/1500
251/251 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1629 - mae: 0.1094 - val_loss: 0.1533 - val_mae: 0.1045
Epoch 4/1500
251/251 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.1514 - mae: 0.0959 - val_loss: 0.1472 - val_mae: 0.0973
Epoch 5/1500
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.1375 - mae: 0.0898 - val_loss: 0.1271 - val_mae: 0.0868
Epoch 6/1500
251/251 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1271 - mae: 0.0882 - val_loss: 0.1190 - val_mae: 0.0734
Epoch 7/1500
251/251 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1226 - mae: 0.0865 - val_loss: 0.1134 - val_mae: 0.0789
Epoch 8/1500
251/251 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.1160 - mae: 0.0807 - val_loss: 0.1119 - val_mae: 0.0726
Epoch 9/1500
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/

In [15]:
# ==========================================
# 5. EVALUATION AND PLOTTING
# ==========================================
print("\n--- Model Evaluation ---")

# Predict on validation data
y_pred = model.predict(val_dataset)

# Plot True vs Predicted for b/a and c/a
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot b/a
axes[0].scatter(y_val[:, 0], y_pred[:, 0], alpha=0.5, c='blue')
axes[0].plot([0, 1], [0, 1], 'r--') # Perfect prediction line
axes[0].set_title("Target: b/a")
axes[0].set_xlabel("True b/a")
axes[0].set_ylabel("Predicted b/a")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

# Plot c/a
axes[1].scatter(y_val[:, 1], y_pred[:, 1], alpha=0.5, c='green')
axes[1].plot([0, 1], [0, 1], 'r--') # Perfect prediction line
axes[1].set_title("Target: c/a")
axes[1].set_xlabel("True c/a")
axes[1].set_ylabel("Predicted c/a")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


--- Model Evaluation ---
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step


In [16]:
T_val = (1 - y_val[:, 0]**2) / (1 - y_val[:, 1]**2)
T_pred = (1 - y_pred[:, 0]**2) / (1 - y_pred[:, 1]**2)

fig, axes = plt.subplots(1, 1, figsize=(6, 5))
axes.scatter(T_val, T_pred, alpha=0.5, c='blue')
axes.plot([0, 1], [0, 1], 'r--')
axes.set_title("Target: T")
axes.set_xlabel("True T")
axes.set_ylabel("Predicted T")

plt.tight_layout()
plt.show()


In [15]:
def plot_training_history(history):
    """
    Plots the training and validation accuracy and loss curves.
    """
    acc = history.history['mae']
    val_acc = history.history['val_mae']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy', marker='o')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='o')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss', marker='o')
    plt.plot(epochs_range, val_loss, label='Validation Loss', marker='o')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

In [16]:
plot_training_history(history)

In [17]:
results = pd.DataFrame({'true_b_a': y_val[:, 0], 'pred_b_a': y_pred[:, 0], 'true_c_a': y_val[:, 1], 'pred_c_a': y_pred[:, 1]})

In [18]:
results.to_csv('predictions_3.csv', index=False, header=True)

In [19]:
results['pred_c_a'] > results['pred_b_a']

0       False
1       False
2       False
3       False
4       False
        ...  
2002    False
2003    False
2004    False
2005    False
2006    False
Length: 2007, dtype: bool

In [21]:
raise(NotImplementedError)

NotImplementedError: 

In [20]:
model.save('model_3.keras')

In [ ]:
model.save_weights('weights_3.')

In [11]:
model = tf.keras.models.load_model('model_3.keras',
                                   custom_objects={
                                    'PearsonHuberLoss': PearsonHuberLoss
                                    })

In [14]:
model = tf.keras.models.load_model('best_tuned_ifu_cnn.keras',
                                   custom_objects={
                                    'PearsonHuberLoss': PearsonHuberLoss
                                    })

/home/astrolander/.generalvenv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 26 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
